In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize
from models import CRW
from tqdm import tqdm

In [ ]:
#plot settings
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}"
})
plt.rcParams["font.size"] = 15
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
figsize = (12,9)

In [ ]:
# generate 1000 CRW polymers of length 10-1000, and record their statistics
low=10
high=1000

num_crw = 1000

rng = np.random.default_rng()
crw_list = [CRW(int(10**rng.uniform(np.log10(low),np.log10(high)))) for i in tqdm(range(num_crw))]

In [ ]:
print(crw_list)

In [ ]:
crw_list[1].show()

In [ ]:
lengths = [crw.n for crw in tqdm(crw_list)]
R_gs = [float(crw.compute_rg()) for crw in tqdm(crw_list)]

print(lengths)
print(R_gs)

In [ ]:
mean_R_gs = np.nanmean()

plt.figure()
plt.scatter(lengths, R_gs, s=1, label="Radius of Gyration")
plt.ylabel("Radius of Gyration")
plt.xlabel("Number of Residues")
plt.xscale("log")
plt.yscale("log")
plt.title(r"$R_g(n)$ for Unfolded RW Polymer")
plt.savefig("RgVsN.png")

In [ ]:
n_max = max(crw.n for crw in crw_list)

ns = np.arange(2, n_max+1)
Rg_ns = np.full((num_crw, len(ns)), np.nan)

plt.figure()
plt.title(r"$\langle R_g(n)\rangle$ (Unfolded RW Polymer)")
plt.xlabel("Subchain length")
plt.ylabel(r"$\langle R_g(n) \rangle$")
plt.xscale("log")
plt.yscale("log")
for crw_idx, crw in enumerate(tqdm(crw_list)):
    
    valid_ns = ns[ns < crw.n]
     
    for n_idx, n in enumerate(valid_ns):
        Rg_ns[crw_idx, n_idx]=crw.compute_rg_n(n)
    plt.plot(valid_ns, Rg_ns[crw_idx, :len(valid_ns)], c="b", alpha=.1)

mean_Rg_n = np.nanmean(Rg_ns, axis=0)
Rg_lo, Rg_hi = np.nanpercentile(Rg_ns, [16, 84], axis=0)

plt.plot(ns, mean_Rg_n, c="r", linestyle="--",label=r"$\langle R_g(n) \rangle$")
plt.fill_between(ns, Rg_lo, Rg_hi, color="r", alpha=10/num_crw)
plt.legend()
plt.savefig("Rg_n.png")
plt.show()

In [ ]:
# Now when the polymers begin to fold what happens?
# First examine different central forces, and what they do to to final R_g(N) and the <R_g(n)> at different temperatures